## Preamble

In [1]:
%pip install pycountry-24.6.1-py3-none-any.whl

Processing ./pycountry-24.6.1-py3-none-any.whl
Note: you may need to restart the kernel to use updated packages.


In [2]:
import IPython

# restart python kernel
IPython.Application.instance().kernel.do_shutdown(restart=True)

{'status': 'ok', 'restart': True}

In [1]:
try:
    print(f"Spark version {sc.version}")
except:
    %run ../spark-instance.ipynb

SparkConf created
Started SparkSession
Spark version 3.5.3
You should be able to access the Spark UI at: https://dacs-compute-gate.ewi.utwente.nl:9999/user/g.luvizottocesar@utwente.nl/proxy/4040/stages/
Note that you may have to Enable extensions first via the Extension Manager.


In [2]:
clean_spark()

In [2]:
from datetime import datetime, timedelta, date
from dateutil import rrule
import re
import itertools
import logging

from collections import defaultdict
import publicsuffixlist as psl
import pandas as pd
import networkx as nx
from graphframes import GraphFrame
from functools import reduce
import pycountry

import pyspark.sql.types as pst
import pyspark.sql.functions as psf
from pyspark.sql.window import Window

import matplotlib.pyplot as plt

In [3]:
# Create logger
logger = logging.getLogger("dmarc")
logger.setLevel(logging.INFO)

# Prevent duplicate logs if logger already has handlers
if not logger.handlers:
    # Create file handler
    file_handler = logging.FileHandler("logs.log", encoding='utf-8')
    file_handler.setLevel(logging.INFO)
    
    # Create formatter
    formatter = logging.Formatter(
        fmt='%(asctime)s %(levelname)-8s %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Add formatter to handler
    file_handler.setFormatter(formatter)
    
    # Add handler to logger
    logger.addHandler(file_handler)
else:
    print("logger already exist!")

# Optionally, prevent propagation to root logger to avoid duplicates
logger.propagate = False

In [4]:
# Base prefix of fDNS warehouse data
FDNS_WAREHOUSE_BASE = "category=fdns/type=warehouse"
# Prefix format() template for (source, date)-partition fDNS warehouse data
FDNS_SOURCE_DT_PARTITION_FMT_TEMPLATE = os.path.join(FDNS_WAREHOUSE_BASE, "source={source}", "year={year}", "month={month:02d}", "day={day:02d}")

sources = ["com", "net", "org", "czds", "alexa", "majestic", "opencc", "radar", "tranco", "umbrella", "crux"]

# additional sources:
# infra:mx, infra:ns
# alexa is old; not found in our recent data set

In [19]:
psl_obj = psl.PublicSuffixList()
broadcast_psl_obj = spark.sparkContext.broadcast(psl_obj)

tld_regex_dict = {}
for tld in psl.PublicSuffixList()._publicsuffix:
    tld_escaped = tld.replace("*.", "").replace("!", "")
    _domain_pattern = fr'([^.]+)[.]({tld_escaped}).?$'
    tld_regex_dict[tld_escaped] = re.compile(_domain_pattern, re.IGNORECASE)  
broadcast_tld_regex = spark.sparkContext.broadcast(tld_regex_dict)

@psf.pandas_udf(pst.StringType())
def extract_apex_domain(query_name: pd.Series) -> pd.Series:
    _psl_obj = broadcast_psl_obj.value
    _tld_regex_dict = broadcast_tld_regex.value
    def _extract_apex_domain(x: str):
        if x is None:
            return None

        # invalid case:query:_dmarc.siter.io., txt_text contains ma""ilinblue.com!10m
        # invalid case:query:_dmarc.fnspfdr.sk, txt_text contains rua=mailto:dmarc@hostcreators.sk ruf=mailto:dmarc@hostcreators.sk; resulting in a rua_domain:hostcreators.sk hostcreators.sk
        if '"' in x or ' ' in x:
            return None

        tld = _psl_obj.publicsuffix(x)  # get the com from a.example.com
        if tld is None:
            return None  # invalid suffix

        tld_escaped = tld.replace("*.", "")
        x_escaped = x
        # rfc7489#section-6.2
        #   For example, the URI "mailto:reports@example.com!50m" would request
        #   that a report be sent via email to "reports@example.com" so long as
        #   the report payload does not exceed 50 megabytes.
        if "!" in tld_escaped:
            tld_escaped = ''.join(tld_escaped.split("!")[0])
            x_escaped = ''.join(x.split("!")[0])
            
        try:
            prog = _tld_regex_dict[tld_escaped]
        except KeyError:
            logger.error(f"Error parsing: {x}")
            return None

        m = prog.search(x_escaped)
        if m:
            return f"{m.group(1).strip()}.{m.group(2).strip()}"
        return None
    return query_name.apply(_extract_apex_domain)


@psf.pandas_udf(pst.StringType())
def extract_cctld(query_name: pd.Series) -> pd.Series:
    # extract the .br from example.com.br because we don't need a second level tld to identify a domain from a country
    # except for .gov domains, where we explicitly want the .gov.br domain to know we're talking about domains from government
    _psl_obj = broadcast_psl_obj.value
    def _extract_cctld(x):
        tld = _psl_obj.publicsuffix(x)
        if '.gov' in x:
            return tld
        return x.split(".")[-2]
    return query_name.apply(_extract_cctld)


@psf.pandas_udf(pst.BooleanType())
def is_gov_domain(query_name: pd.Series) -> pd.Series:
    _psl_obj = broadcast_psl_obj.value
    def _is_gov_domain(x):
        # Handle null/None values
        if pd.isna(x) or x is None:
            return False
        # Convert to string if needed
        #x = str(x)
        tld = _psl_obj.publicsuffix(x)
        return 'gov' in tld and not 'amazonaws' in tld
    return query_name.apply(_is_gov_domain)


def dmarc_filter(df):
    return df.filter(
        (psf.col("query_type") == "TXT")
        & (psf.col("query_name").startswith("_dmarc"))
        & (psf.col("txt_text").rlike(r'^"v\s*=\s*DMARC1\s*;'))
    )

# claude sonet regex
# old: r'rua\s*=\s*([^";]*)' older: "rua=[^@]+@([^;\s[\"]]+)"
# (?:^|;) is a starting point of a non-matching group. either you have multiple tags but then you close the previous with ; or you are just starting the tag (the only tag)
rua_pattern = r'(?:^|;)\s*rua\s*=\s*([^\";]+)'  # the [^\";] means that if " or ; is found before closing the tag, then the tag is invalid
ruf_pattern = r'(?:^|;)\s*ruf\s*=\s*([^\";]+)'  # valid example: r'ruf   =    ReJeCt  ;'. So, exact match key, case insensitive value, with many or without whitespace (\s*) surrounding = and ; where the ; is the last element
user_domain_pattern = r'[^@,\s]*@'  # matches the user from the email domain. Example: r'user@domain.com' matches r'user@'
def extract_rua_ruf_domains(df):
    return dmarc_filter(df  # filter for dmarc...
    ).withColumn("rua_raw", psf.regexp_replace(psf.regexp_extract(psf.col("txt_text"), rua_pattern, 1), user_domain_pattern, "")
    ).withColumn("ruf_raw", psf.regexp_replace(psf.regexp_extract(psf.col("txt_text"), ruf_pattern, 1), user_domain_pattern, "")
    ).withColumn("rua_raw", psf.regexp_replace(psf.col("rua_raw"), 'mailto:', '')  # trim mailto:
    ).withColumn("ruf_raw", psf.regexp_replace(psf.col("ruf_raw"), 'mailto:', '')  # trim mailto:
    ).withColumn(# Split RUA by comma
        "rua_domains",
        psf.split(psf.col("rua_raw"), r'\s*,\s*')
    ).withColumn(
        "ruf_domains",
        psf.split(psf.col("ruf_raw"), r'\s*,\s*')
    ).withColumn("rua_domains",  # Trim each element
        psf.transform(psf.col("rua_domains"), lambda x: psf.trim(x))
    ).withColumn("ruf_domains", 
        psf.transform(psf.col("ruf_domains"), lambda x: psf.trim(x))
    ).withColumn("rua_domains",   # Filter out empty strings
        psf.filter(psf.col("rua_domains"), lambda x: psf.length(x) > 0)
    ).withColumn("ruf_domains", 
        psf.filter(psf.col("ruf_domains"), lambda x: psf.length(x) > 0)
    ).drop(*("rua_raw", "ruf_raw"))  # remove intermediary columns


# following rua/ruf regex idea
p_pattern = r'(?:^|;)\s*p\s*=\s*([^\";]+)'  # we capture any value in the tag to evaluate later whether is invalid or not
sp_pattern= r'(?:^|;)\s*sp\s*=\s*([^\";]+)'
adkim_pattern = r'(?:^|;)\s*adkim\s*=\s*([^\";]+)'
aspf_pattern = r'(?:^|;)\s*aspf\s*=\s*([^\";]+)'
def extract_dmarc_tags(df):
    return extract_rua_ruf_domains(  # already filter for dmarc...
        df.withColumn(  
            # trim because it allows whitespace around; lower case because it is case insensitive
            "p", psf.lower(psf.trim(psf.regexp_extract(psf.col("txt_text"), p_pattern, 1)))
        ).withColumn(
            "sp", psf.lower(psf.trim(psf.regexp_extract(psf.col("txt_text"), sp_pattern, 1)))
        ).withColumn(
            "adkim", psf.lower(psf.trim(psf.regexp_extract(psf.col("txt_text"), adkim_pattern, 1)))
        ).withColumn(
            "aspf", psf.lower(psf.trim(psf.regexp_extract(psf.col("txt_text"), aspf_pattern, 1)))
        ).withColumn(  # assigning valid values, invalid or empty for each tag
            "p", psf.when(psf.col("p").isin(["none", "quarantine", "reject"]), psf.col("p"))
                 .when((psf.col("p").isNull()) | (psf.length(psf.col("p")) == 0), None)
                 .otherwise("invalid")
        ).withColumn(
            "sp", psf.when(psf.col("sp").isin(["none", "quarantine", "reject"]), psf.col("sp"))
                 .when((psf.col("sp").isNull()) | (psf.length(psf.col("sp")) == 0), None)
                 .otherwise("invalid")
        ).withColumn(
            "adkim", psf.when(psf.col("adkim").isin(["r", "s"]), psf.col("adkim"))
                 .when((psf.col("adkim").isNull()) | (psf.length(psf.col("adkim")) == 0), None)
                 .otherwise("invalid")
        ).withColumn(
            "aspf", psf.when(psf.col("aspf").isin(["r", "s"]), psf.col("aspf"))
                 .when((psf.col("aspf").isNull()) | (psf.length(psf.col("aspf")) == 0), None)
                 .otherwise("invalid")
        )
    )

In [ ]:
# deprecated function
if False:
    # when using r"" string (raw), no need to escape characters ("\\.")
    # regex to extract sld.tld
    all_suffixes = [s for s in psl.PublicSuffixList()._publicsuffix]  #if s and s.isascii()
    tld_pattern = '|'.join(all_suffixes).replace("*.", "") #.replace(".", "\\.")
    
    #domain_pattern_dot = fr"([^.]+)[.]({tld_pattern})[.]$"
    domain_pattern = fr"([^.]+)[.]({tld_pattern}).?$"
    
    #).withColumn("sld", psf.regexp_extract(psf.col("query_name"), domain_pattern, 1)
    #).withColumn("tld", psf.regexp_extract(psf.col("query_name"), domain_pattern, 2)
    #).withColumn("apex_domain", psf.concat_ws(".", psf.col("sld"), psf.col("tld"))  # psf.lit("") # last psf.lit("") is to add . to the end of the string
    
    # without considering termination "."
    #domain_pattern = f"([^.]+)\\.({tld_pattern})$"  # use [\\.] or \\.; doesn't matter...
    #domain_pattern = fr"([^.]+)[.]({tld_pattern})$"
    def get_apex_domain(df, extract_column: str="query_name", pattern: str=domain_pattern, new_column_name: str="apex_domain"):
        # extract_column can be any column with a single domain name (not a list of domain names)
        df = df.withColumn(
            "sld", psf.regexp_extract(psf.col(extract_column), pattern, 1)
        ).withColumn("tld", psf.regexp_extract(psf.col(extract_column), pattern, 2)
        ).withColumn(new_column_name, 
                psf.when((psf.col("tld").isNull()) | (psf.length(psf.col("tld")) == 0), "")
                   .otherwise(psf.concat_ws(".", psf.col("sld"), psf.col("tld"), psf.lit("")))  # the last psf.lit("") will add "." in the end of the domain.
        )
    
        return df

### Code examples

In [ ]:
# TEST CODE - EXTRACTED FROM EXAMPLE v0.1
SOURCES = ["com", "net", "org", "czds", "alexa", "majestic", "opencc", "radar", "tranco", "umbrella", "infra:ns", "infra:mx"]
START_DATE = datetime(2024, 1, 11)
END_DATE   = START_DATE

for i_dt in rrule.rrule(rrule.DAILY, dtstart=START_DATE, until=END_DATE):

    i_dt_sources_df = spark.read.option(
        "basePath", "s3a://openintel/{base}".format(base = FDNS_WAREHOUSE_BASE)
    ).format("parquet").load([
        "s3a://openintel/{prefix}".format(
            prefix = FDNS_SOURCE_DT_PARTITION_FMT_TEMPLATE.format(
                source = i_source, year = i_dt.year, month = i_dt.month, day = i_dt.day
            )
        )
        for i_source in SOURCES
    ])

In [ ]:
# TEST CODE - EXTRACTED FROM EXAMPLE v0.1
tranco_df = spark.read.option(
    "basePath", "s3a://openintel/category=fdns/type=warehouse/"
).parquet(*[ # n.b.: unpack
    "s3a://openintel/category=fdns/type=warehouse/source=tranco/year=2022/month=12/day=01/",
    "s3a://openintel/category=fdns/type=warehouse/source=tranco/year=2023/month=01/day=01"
]).filter(
    psf.col("ns_address").isNotNull()
).groupby(
    "year", "month", "day",
    "ns_address"
).agg(
    psf.countDistinct("query_name").alias("2ld_count") # n.b.: implied, as we only do NS queries on the @
).withColumn(
    "rank",
    psf.row_number().over(psw.Window().partitionBy("year", "month", "day").orderBy(psf.col("2ld_count").desc()))
)

tranco_df.filter(
    (psf.col("rank") <= 5)
).orderBy("year", "month", "day", "rank").show(50, truncate=False)

## Prepare input data for new measurements

In [46]:
# NO NEED TO USE infra:mx;
# a domain doesn't necessarily need to have mx record to be able to send reports
# if the domain sends dmarc report to itself, then it should have an mx record in its dns.
source = "infra:mx"
spark.read.option(
    "basePath", f"s3a://openintel/{FDNS_WAREHOUSE_BASE}/source={source}"
).parquet(f"s3a://openintel/{FDNS_WAREHOUSE_BASE}/source={source}/year=2025"
).select("month", "day").distinct().sort("month", "day", ascending=False
).show(truncate=False)

mx_date = datetime(2025, 7, 11)  # infra:mx has daily measurements; get latest; see previous cell output.
mx_df = spark.read.option(
    "basePath", f"s3a://openintel/{FDNS_WAREHOUSE_BASE}"
).parquet(f"s3a://openintel/{FDNS_WAREHOUSE_BASE}/source={source}/year={mx_date.year}/month={mx_date.month:02d}/day={mx_date.day:02d}")

mx_df.select("query_name", "query_type", "mx_address").show(5, truncate=False)

+-----+---+
|month|day|
+-----+---+
|7    |10 |
|7    |9  |
|7    |8  |
|7    |7  |
|7    |6  |
|7    |5  |
|7    |4  |
|7    |3  |
|7    |2  |
|7    |1  |
|6    |30 |
|6    |29 |
|6    |28 |
|6    |27 |
|6    |26 |
|6    |25 |
|6    |24 |
|6    |23 |
|6    |22 |
|6    |21 |
+-----+---+
only showing top 20 rows



In [84]:
# "You'd have to join back the infra:mx query_name AS mx_address on source=* & response_type=MX & mx_address=mx_address and then get source=*'s _dmarc.query_name"
start_date = datetime(2025, 7, 11)

# extract rua/ruf tags
for source in sources:
    # load dataset
    try:
        source_df = spark.read.option(
            "basePath", f"s3a://openintel/{FDNS_WAREHOUSE_BASE}"
        ).parquet(f"s3a://openintel/{FDNS_WAREHOUSE_BASE}/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}")
    except Exception as e:
        print(f"Path/data for source {source} not found. {e}")
        continue

    if False:
        # remove the mx from the join..
        mx_selected_df = mx_df.select(psf.col("query_name").alias("query_name_mx"))
    
        # join the loaded dataset with the mail server dataset on mx_address names
        # extract rua/ruf addresses from only those with _dmarc queries on TXT record
        mx_on_source_df = source_df.select("query_name", "query_type", "mx_address"
        ).filter(
            (psf.col("query_type") == "MX")
            & (psf.col("mx_address").isNotNull())
        ).join(
            mx_selected_df,
            source_df.mx_address == mx_selected_df.query_name_mx,
            how="inner"
        ).select(psf.col("query_name"))
    
        mx_on_source_dmarc_df = mx_on_source_df.select(psf.concat(psf.lit("_dmarc."), psf.col("query_name")).alias("query_name"))
        
        joined_df = source_df.select("query_name", "query_type", "txt_text").filter(
            (psf.col("query_name").startswith("_dmarc."))
            & (psf.col("query_type") == "TXT")
        ).join(
            mx_on_source_dmarc_df,
            on=["query_name"],
            how="inner"
        ).select("query_name", "query_type", "txt_text")
        df = joined_df

    df = source_df.select("query_name", "query_type", "txt_text").filter(
            (psf.col("query_name").startswith("_dmarc"))
            & (psf.col("query_type") == "TXT")
            & (psf.col("txt_text").contains("v=DMARC1"))
    )

    # rua/ruf tags should be FQDN, not just apex domain, otherwise subsequente measurements won't work.
    rua_ruf_df = extract_rua_ruf_domains(df).filter(
        (psf.length(psf.col("rua_raw")) > 0) | (psf.length(psf.col("ruf_raw")) > 0)
    ).drop(*("query_type", "txt_text")
    ).select("query_name", psf.explode_outer("rua_domains").alias("rua_domain"), psf.explode_outer("ruf_domains").alias("ruf_domain")
    ).withColumn("source", psf.lit(source)
    ).withColumn(
        "query_name", psf.regexp_replace("query_name", r"^_dmarc.", "")  # remove _dmarc. to ease the new measurements
    )

    rua_df = rua_ruf_df.select("source", "query_name", psf.col("rua_domain").alias("rua_ruf_domain"))
    ruf_df = rua_ruf_df.select("source", "query_name", psf.col("ruf_domain").alias("rua_ruf_domain"))

    output = f"s3a://luvizottocesarg-tmp/oi-warehouse/rua-ruf-address/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}"
    rua_df.union(ruf_df).distinct().write.parquet(output)

Path/data for source alexa not found. [PATH_NOT_FOUND] Path does not exist: s3a://openintel/category=fdns/type=warehouse/source=alexa/year=2025/month=07/day=11.


In [85]:
for source in sources:
    try:
        print(
            source, 
            spark.read.option(
                    "basePath", "s3a://luvizottocesarg-tmp/oi-warehouse/rua-ruf-address"
            ).parquet(f"s3a://luvizottocesarg-tmp/oi-warehouse/rua-ruf-address/source={source}/").count()
        )
    except:
        continue

com 14487860
net 996161
org 1061799
czds 2959627
majestic 475134
opencc 631122
radar 514963
tranco 502068
umbrella 187719


## Prepare the dataset

NOT NECESSARY!  
Store a dataset such as:  

source,query_name,apex_domain,country,timestamp,mx_apex_domain,mx_country,rua_domain,rua_apex_domain,ruf_domain,ruf_apex_domain,has_passed_edv,is_override_allowed,override_rua_domain,override_ruf_domain  

In [17]:
# country is taken from A/AAAA records. 
d = datetime(2025, 7, 11)
spark.read.option(
    "basePath", f"s3a://openintel/{FDNS_WAREHOUSE_BASE}"
).parquet(f"s3a://openintel/{FDNS_WAREHOUSE_BASE}/source=czds/year={d.year}/month={d.month:02d}/day={d.day:02d}"
).filter(psf.col("country").isNotNull()).groupBy("query_type").count().show()

+----------+--------+
|query_type|   count|
+----------+--------+
|     AFSDB|     693|
|         A|96480432|
|      AAAA|33814238|
+----------+--------+



In [82]:
# THIS DOES NOT WORK! IT IS TOO HEAVY AND THE TASK FAILS. 
# A BETTER APPROACH IS TO BUILD DIFFERENT GRAPHS;
# 1 - WITH DOMAIN NAMES RELATIONSHIPS
# 2A - TLDS 
# 2B - CHECK WHETHER WE MEASURE A RECORDS OF NAMES IN RUA/RUF TAGS AND GET THE ASSOCIATED COUNTRY
start_date = datetime(2025, 7, 11)

# extract rua/ruf tags
for source in sources:
    # load dataset
    try:
        source_df = spark.read.option(
            "basePath", f"s3a://openintel/{FDNS_WAREHOUSE_BASE}"
        ).parquet(f"s3a://openintel/{FDNS_WAREHOUSE_BASE}/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}")
    except Exception as e:
        print(f"Path/data for source {source} not found. {e}")
        continue
    
    mx_df = spark.read.option("basePath", f"s3a://openintel/{FDNS_WAREHOUSE_BASE}"
        ).parquet(f"s3a://openintel/{FDNS_WAREHOUSE_BASE}/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}")

    mx_selected_df = mx_df.select(
        psf.col("query_name").alias("query_name_mx"),
        psf.col("country").alias("mx_country")
    ).withColumn("mx_apex_domain", extract_apex_domain("query_name_mx"))

    # join the loaded dataset with the mail server dataset on mx_address names
    # extract rua/ruf addresses from only those with _dmarc queries on TXT record
    mx_on_source_df = source_df.select("query_name", "query_type", "mx_address"
    ).filter(
        (psf.col("query_type") == "MX")
        & (psf.col("mx_address").isNotNull())
    ).join(
        mx_selected_df,
        source_df.mx_address == mx_selected_df.query_name_mx,
        how="left"
    ).withColumn(
        "apex_domain", extract_apex_domain("query_name")
    ).select("apex_domain", "mx_country", "mx_apex_domain").distinct()  # get uniqs 

    dmarc_names_df = extract_rua_ruf_domains(source_df.filter(
            (psf.col("query_name").startswith("_dmarc"))
            & (psf.col("query_type") == "TXT")
            & (psf.col("txt_text").contains("v=DMARC1"))
    ).withColumn(
        "apex_domain", extract_apex_domain("query_name")
    ).select("apex_domain", "query_name", "query_type", "txt_text"
    )).select("apex_domain", 
              psf.explode_outer("rua_domains").alias("rua_domain"), 
              psf.explode_outer("ruf_domains").alias("ruf_domain")
    ).groupBy("apex_domain").agg(
        psf.collect_set("rua_domain").alias("rua_domains"),
        psf.collect_set("ruf_domain").alias("ruf_domains")
    )

    dmarc_country_df = source_df.select("query_name", "query_type", "country", "timestamp"
    ).filter(psf.col("query_type") == "A"
    ).withColumn(
        "apex_domain", extract_apex_domain("query_name")
    ).join(
        dmarc_names_df,
        on=["apex_domain"],
        how="right"
    ).select("apex_domain", "country", "timestamp", "rua_domains", "ruf_domains")

    dmarc_compress_df = dmarc_country_df.join(
        mx_on_source_df,
        on=["apex_domain"],
        how="left"
    ).withColumn(
        "source", psf.lit(source)
    )

    # source,query_name,apex_domain,country,timestamp,mx_apex_domain,mx_country,rua_domain,rua_apex_domain,ruf_domain,ruf_apex_domain
    _dmarc_df = dmarc_compress_df.select(
        "apex_domain", "country", "timestamp", "mx_apex_domain", "mx_country",
        psf.explode_outer("rua_domains").alias("rua_domain"),
        psf.explode_outer("ruf_domains").alias("ruf_domain"),
    ).withColumn(
        "rua_apex_domain", extract_apex_domain("rua_domain")
    ).withColumn(
        "ruf_apex_domain", extract_apex_domain("ruf_domain")
    ).distinct()  #.dropDuplicates("apex_domain", "mx_apex_domain", "rua_apex_domain", "ruf_apex_domain")  # no duplicates to make the graph

    break
    output = f"s3a://luvizottocesarg-tmp/oi-warehouse/dmarc-dataset/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}"
    _dmarc_df.write.parquet(output)

## Loading the dataset

In [20]:
source = "tranco"
start_date = datetime(2025, 7, 11)  # tranco has daily measurements

tranco_df = spark.read.option(
    "basePath", f"s3a://openintel/{FDNS_WAREHOUSE_BASE}"
).parquet(f"s3a://openintel/{FDNS_WAREHOUSE_BASE}/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}")

In [21]:
cctlds = set({})
for country in pycountry.countries:
    cctlds.add(f"{country.alpha_2.lower()}")

print(len(cctlds))

for i in psl_obj._publicsuffix:
    tld = psl_obj.publicsuffix(i)
    if 'gov' in tld and not 'amazonaws' in tld:
        cctlds.add(i)

print(len(cctlds))

cctld_df = spark.createDataFrame([(cctld,) for cctld in cctlds], ["tld"])
cctld_df.show(3, truncate=False)

249
491
+------+
|tld   |
+------+
|gov.mw|
|gov.dz|
|ch    |
+------+
only showing top 3 rows



In [8]:
cctld_df.printSchema()

root
 |-- tld: string (nullable = true)



In [9]:
tranco_df.printSchema()

root
 |-- query_type: string (nullable = true)
 |-- query_name: string (nullable = true)
 |-- response_type: string (nullable = true)
 |-- response_name: string (nullable = true)
 |-- response_ttl: integer (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- rtt: double (nullable = true)
 |-- worker_id: integer (nullable = true)
 |-- status_code: integer (nullable = true)
 |-- ad_flag: integer (nullable = true)
 |-- section: string (nullable = true)
 |-- ip4_address: string (nullable = true)
 |-- ip6_address: string (nullable = true)
 |-- country: string (nullable = true)
 |-- as: string (nullable = true)
 |-- as_full: string (nullable = true)
 |-- ip_prefix: string (nullable = true)
 |-- cname_name: string (nullable = true)
 |-- dname_name: string (nullable = true)
 |-- mx_address: string (nullable = true)
 |-- mx_preference: integer (nullable = true)
 |-- ns_address: string (nullable = true)
 |-- txt_text: string (nullable = true)
 |-- ds_key_tag: integer (nullable = true)
 

In [10]:
# cc of domain: response_type A for query_name
# cc of mx: response_type A for mx_address in infra:mx
# mx of domain = response_type MX for query_name
# for rua/ruf extract via regex -- psf.regex_extract

## Data overview

In [8]:
# be careful with what to consider "all domains";
# there are more A queries than MX queries...
# queries might also differ (e.g. _dmarc.fqdn on txt type or fqdn on mx type)
print(
    tranco_df.filter(
        psf.col("query_type") == "A"
    ).select("query_name").distinct().count(),

    tranco_df.filter(
        psf.col("query_type") == "MX"
    ).select("query_name").distinct().count(),
)

1756442 688056


Nr of ccTLDs

Nr of .gov related domains

In [14]:
extract_dmarc_tags(tranco_df).withColumn("apex_domain", extract_apex_domain("query_name")).dropDuplicates(["apex_domain"]).select("query_name", "apex_domain"
).withColumn(
    "tld", extract_cctld("query_name")
).filter(
    psf.col("tld").contains("gov")
).count()

5934

Nr of ccTLDs domains

In [43]:
extract_dmarc_tags(tranco_df).withColumn("apex_domain", extract_apex_domain("query_name")).dropDuplicates(["apex_domain"]).select("query_name", "apex_domain"
).withColumn(
    "tld", extract_cctld(psf.col("query_name"))  #psf.concat(psf.lit("."), psf.element_at(psf.split(psf.col("query_name"), "\\."), -2))  # -2 because the query always finishes with .; .com.
).join(
    cctld_df, on=["tld"], how="inner"
).count()

163109

All domains that exists

In [26]:
all_domains_cnt = tranco_df.withColumn("apex_domain", extract_apex_domain("query_name")).select("apex_domain").distinct().count()

print(all_domains_cnt)
# tranco (2024, 3, 14): 943465
# tranco (2025, 7, 11): 1005207

1004190


In [23]:
try:
    dmarc_domains_df.unpersist()
except:
    pass

dmarc_domains_df = dmarc_filter(tranco_df
).withColumn("apex_domain", extract_apex_domain("query_name")).select("apex_domain").distinct()

dmarc_domains_df.persist()

DataFrame[apex_domain: string]

Domains with DMARC (v=DMARC1)

In [24]:
dmarc_domains_cnt = dmarc_domains_df.count()

In [27]:
print("w/ dmarc:", dmarc_domains_cnt, "\nproportion of domains w/ DMARC w.r.t all domains:", round(dmarc_domains_cnt/all_domains_cnt*100, 1), "%")

w/ dmarc: 424716 
proportion of domains w/ DMARC w.r.t all domains: 42.3 %


Domains that have an MX server

In [15]:
names_with_mx_df = tranco_df.filter(
        (psf.col("query_type") == "MX")
        & (psf.col("mx_address").isNotNull())
).withColumn("apex_domain", extract_apex_domain("query_name")).select("apex_domain").distinct()

In [23]:
names_w_mx_cnt = names_with_mx_df.count()
print(names_w_mx_cnt)

677749


In [24]:
print(round(names_w_mx_cnt/all_domains_cnt*100,1))

67.4


proportion of domains that have MX and have DMARC enabled (v=DMARC1) and w/ MX

In [17]:
dmarc_domains_w_mx_df = dmarc_domains_df.join(
    names_with_mx_df, on=["apex_domain"], how="inner"
).select("apex_domain").distinct()

dmarc_domains_w_mx_cnt = dmarc_domains_w_mx_df.count()

In [18]:
print("w/ dmarc & mx:", dmarc_domains_w_mx_cnt, "\nproportion of domains w/ DMARC and MX w.r.t domains w/ MX only:", round(dmarc_domains_w_mx_cnt/names_w_mx_cnt*100, 1), "%")

w/ dmarc & mx: 402763 
proportion of domains w/ DMARC and MX w.r.t domains w/ MX only: 59.4 %


without MX but with DMARC

In [25]:
no_mx_w_dmarc = dmarc_domains_cnt-dmarc_domains_w_mx_cnt
print(no_mx_w_dmarc, "proportion w.r.t. all domains:", round(no_mx_w_dmarc/all_domains_cnt*100,1))

23242 proportion w.r.t. all domains: 2.3


What is the proportion of domains using well formatted rua/ruf tags with valid domains?

In [22]:
dmarc_tag_df = extract_dmarc_tags(tranco_df).withColumn(
    "apex_domain", extract_apex_domain("query_name")
).select("query_name", "apex_domain", "p", "sp", "aspf", "adkim", "rua_domains", "ruf_domains")

try:
    dmarc_df.unpersist()
except:
    pass

# one domain may have multiple TXT records; then explode the lists and join them again...
dmarc_df = dmarc_tag_df.select(
    "query_name",
    "apex_domain",
    "p", "sp", "aspf", "adkim",
    psf.explode_outer(psf.col("ruf_domains")).alias("ruf_domain"),
    psf.explode_outer(psf.col("rua_domains")).alias("rua_domain")
#).withColumn(
#    "rua_apex_domain", extract_apex_domain("rua_domain")
#).withColumn(
#    "ruf_apex_domain", extract_apex_domain("ruf_domain")
).groupBy("query_name", "apex_domain", "p", "sp", "aspf", "adkim"
).agg(
    psf.collect_set("rua_domain").alias("rua_domains"),
    psf.collect_set("ruf_domain").alias("ruf_domains"),
)

dmarc_df.persist()

DataFrame[query_name: string, apex_domain: string, p: string, sp: string, aspf: string, adkim: string, rua_domains: array<string>, ruf_domains: array<string>]

In [29]:
dmarc_report_address_df = dmarc_df.withColumn(
    "has_rua", psf.when(psf.size(psf.col("rua_domains")) == 0, False).otherwise(True)
).withColumn(
    "has_ruf", psf.when(psf.size(psf.col("ruf_domains")) == 0, False).otherwise(True)
)

either_rua_ruf_nr = dmarc_report_address_df.filter((psf.col("has_rua") == True) | (psf.col("has_ruf") == True)).select("apex_domain").distinct().count()
rua_nr = dmarc_report_address_df.filter(psf.col("has_rua") == True).select("apex_domain").distinct().count()
ruf_nr = dmarc_report_address_df.filter(psf.col("has_ruf") == True).select("apex_domain").distinct().count()

dmarc_domains_cnt = dmarc_df.select("apex_domain").distinct().count()

print(
    "either rua or ruf:", either_rua_ruf_nr, "out of", dmarc_domains_cnt,
    round(either_rua_ruf_nr / dmarc_domains_cnt * 100, 1), "%",

    "\nrua:", rua_nr, "out of", dmarc_domains_cnt,
    round(rua_nr / dmarc_domains_cnt * 100, 1), "%",

    "\nruf:", ruf_nr, "out of", dmarc_domains_cnt,
    round(ruf_nr / dmarc_domains_cnt * 100, 1), "%"
)

either rua or ruf: 282943 out of 424716 66.6 % 
rua: 278668 out of 424716 65.6 % 
ruf: 146862 out of 424716 34.6 %


spoofed paper (p.13): rua: 41.99%, ruf: 20.57%

Comparison with related work:  
[Spoofed] - czds, passive DNS SIE EU, CT logs, Tranco 5M; Table 2: 
NO MX, DMARC with rua/ruf = 1,270,352  
NO MX, DMARC without rua/ruf = 909,347  
MX, DMARC with rua/ruf = 5,435,578  
MX, DMARC without rua/ruf = 8,354,295  

[Ashiq]  
TLD,  Domains with MX Records,DMARC,DMARC %, Report,Report %, Report from Ext.,Report from Ext. %  
.com, 75.6M,                  5.0M, 6.6%,    2.4M, 49.4%,     1.7M, 68.8%  
.net, 6.5M,                   453K, 6.9%,    245K, 54.1%,     172K, 70.2%  
.org, 5.8M,                   390K, 6.7%,    213K, 54.5%,     152K, 71.4%  
.se,  848K,                   81K,  9.6%,    30K,  37.4%,     24K,  80.1%  

In [63]:
dmarc_report_address_df.groupBy("has_rua", "has_ruf").count().withColumn(
        "percent", psf.lit(100) * psf.col('count') / psf.sum('count').over(Window.partitionBy())
).show()

+-------+-------+------+------------------+
|has_rua|has_ruf| count|           percent|
+-------+-------+------+------------------+
|   true|  false|137325|  32.2003892419162|
|   true|   true|144669| 33.92243299645931|
|  false|  false|140280| 32.89328674936103|
|  false|   true|  4196|0.9838910122634652|
+-------+-------+------+------------------+



In [23]:
cctld_dmarc_df = dmarc_df.withColumn(
    "tld", extract_cctld(psf.col("query_name"))
).join(
    cctld_df, on=["tld"], how="inner"
).withColumn(
    "is_gov", is_gov_domain(psf.col("query_name"))
)

In [31]:
output = f"s3a://luvizottocesarg-tmp/oi-warehouse/testing/source=tranco/cctld_dmarc/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}"
cctld_dmarc_df.write.parquet(output)

## RQ1 - How DMARC is used in different countries

See other jupyter notebook in https://github.com/gustavoluvizotto/dmarc-aggr-report-dependency/blob/main/notebooks/process-results.ipynb

## RQ2 - How DMARC reporting is used in different countries

### Extracting graph

In [24]:
# using GraphFrames
def create_graphframes_from_pyspark_df(df):
    # apex_domain = sld.tld (using regex domain_pattern)
    apex_vertices_df = df.select("apex_domain").withColumn("is_gov", is_gov_domain(psf.col("apex_domain"))).distinct()

    rua_vertices_df = df.select(psf.explode_outer("rua_domains").alias("rua_domain")
    ).withColumn(
        "rua_apex_domain", extract_apex_domain(psf.col("rua_domain"))
    ).select(psf.col("rua_apex_domain").alias("apex_domain")).withColumn("is_gov", is_gov_domain(psf.col("apex_domain"))).distinct()

    ruf_vertices_df = df.select(psf.explode_outer("ruf_domains").alias("ruf_domain")
    ).withColumn(
        "ruf_apex_domain", extract_apex_domain(psf.col("ruf_domain"))
    ).select(psf.col("ruf_apex_domain").alias("apex_domain")).withColumn("is_gov", is_gov_domain(psf.col("apex_domain"))).distinct()

    # id (=apex_domain of query_name, rua_domain, ruf_domain), is_gov (=is the domain a governmental domain?
    vertices = apex_vertices_df.union(rua_vertices_df).union(ruf_vertices_df).select(psf.col("apex_domain").alias("id"), "is_gov").distinct()

    # src(=apex_domain), dst(=rua/ruf_apex_domain), edge_type(rua/ruf), label(RUA/RUF)
    # Create edges for RUA relationships
    rua_edges = df.select(psf.col("apex_domain").alias("src"),
                          psf.explode_outer("rua_domains").alias("rua_domain"),
                          psf.lit("rua").alias("edge_type")
        ).withColumn(
            "rua_apex_domain", extract_apex_domain(psf.col("rua_domain"))
        ).filter(psf.col("rua_apex_domain").isNotNull()
        ).select("src", psf.col("rua_apex_domain").alias("dst"), "edge_type").distinct()

    
    # Create edges for RUF relationships
    ruf_edges = df.select(psf.col("apex_domain").alias("src"),
                          psf.explode_outer("ruf_domains").alias("ruf_domain"),
                          psf.lit("ruf").alias("edge_type")
        ).withColumn(
            "ruf_apex_domain", extract_apex_domain(psf.col("ruf_domain"))
        ).filter(psf.col("ruf_apex_domain").isNotNull()
        ).select("src", psf.col("ruf_apex_domain").alias("dst"), "edge_type").distinct()
    
    # Union all edges
    edges = rua_edges.union(ruf_edges)
    
    return GraphFrame(vertices, edges)


G = create_graphframes_from_pyspark_df(cctld_dmarc_df)

In [25]:
# Graph statistics
# for GraphFrame
print(f"Number of nodes: {G.vertices.count()}")
print(f"Number of edges: {G.edges.count()}")

# without filter cctlds on tranco 20250711:
#Number of nodes: 435893
#Number of edges: 346886

Number of nodes: 170993
Number of edges: 176115


In [26]:
num_ruas = G.edges.filter("edge_type = 'rua'").count()
print("The number of RUA edges is", num_ruas)

num_rufs = G.edges.filter("edge_type = 'ruf'").count()
print("The number of RUF edges is", num_rufs)

# without filter cctlds on tranco 20250711:
#The number of RUA edges is 315303
#The number of RUF edges is 158240

The number of RUA edges is 117748
The number of RUF edges is 58367


In [14]:
G.vertices.filter(psf.col("id") == "brevo.com").show(truncate=False)

+---------+------+
|id       |is_gov|
+---------+------+
|brevo.com|false |
+---------+------+



In [27]:
cctld_dmarc_df.groupBy("is_gov").count().show()

+------+------+
|is_gov| count|
+------+------+
|  true|  5958|
| false|158430|
+------+------+



In [28]:
G.vertices.groupBy("is_gov").count().show()  # here includes rua/ruf domains as wel

+------+------+
|is_gov| count|
+------+------+
|  true|  6032|
| false|164961|
+------+------+



Export graph to import in neo4j

In [30]:
# exporting vertices and edges to s3 to load with neo4j
edges_output = f"s3a://luvizottocesarg-tmp/oi-warehouse/aggregation/what=graph_edges/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}"
vertices_output = f"s3a://luvizottocesarg-tmp/oi-warehouse/aggregation/what=graph_vertices/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}"

G.edges.write.parquet(edges_output)
G.vertices.write.parquet(vertices_output)

In [32]:
spark.read.parquet(vertices_output).count()

170993

## RQZ - To what extent are domain owners dependent on third-parties organizations for processing DMARC reports?

OLD: Get DMARC measurements from domains that exist and that have an MX server

In [37]:
# OLD CODE
if False:
    # unpersist what is going to be persisted in this cell in case we run the cell 2x
    try:
        dmarc_w_mx_df.unpersist()
    except:
        pass
    
    # limit mx_names_df to speed up execution
    # a domain that may have dmarc meas and mx server may appear without having them because of this cut;
    limit = 0  # all; no limit
    #limit = 10000
    
    # domains may have 2 TXT answers for _dmarc queries. E.g.:
    # dig -t txt +short _dmarc.vaktija.ba
    #"v=DMARC1; p=reject; sp=reject; adkim=s; aspf=s;"
    #"v=DMARC1; p=none; rua=mailto:postmaster@vaktija.ba"
    apex_df = tranco_df.filter(
        (psf.col("query_type") == "MX")
        | (psf.col("query_type") == "TXT")
        | (psf.col("query_type") == "A")
    ).withColumn(
        "apex_domain", extract_apex_domain(psf.col("query_name"))
    ).select("query_name", "query_type", "apex_domain", "ip4_address", "mx_address", "year", "month", "day", "country", "txt_text")
    
    # apply limit of apex domain to speed up analysis. Comment this code
    if limit == 0:
        df = apex_df
    else:
        lim_distinct_domains = apex_df.select("apex_domain").dropDuplicates().limit(limit)
        lim_apex_df = apex_df.join(lim_distinct_domains, on="apex_domain", how="inner")
        df = lim_apex_df
    
    window = Window.partitionBy("apex_domain")
    
    flags_df = df.withColumn(
        "has_dmarc", psf.max(psf.when(psf.col("query_name").startswith("_dmarc."), 1).otherwise(0)).over(window)
    ).withColumn(
        "has_mx", psf.max(psf.when((psf.col("query_type") == "MX") & psf.col("mx_address").isNotNull(), 1).otherwise(0)).over(window)
    ).withColumn(
        "has_ip", psf.max(psf.when(psf.col("ip4_address").isNotNull(), 1).otherwise(0)).over(window)
    )
    
    # Filter for domains that have both DMARC and MX records
    dmarc_w_mx_df = flags_df.filter(
        (psf.col("has_dmarc") == 1)
        #& (psf.col("has_mx") == 1)
    ).drop("has_dmarc", "has_mx", "has_ip")
    
    #if limit == 0:
    dmarc_w_mx_df.persist()
    
    dmarc_w_mx_df.show(truncate=False)

In [11]:
# limit is non deterministic
dmarc_w_mx_df.count()

42100

In [12]:
# I did some filtering (domains that has_dmarc and has_mx), that's why is not "lim" anymore
dmarc_w_mx_df.select("apex_domain").distinct().count()

3634

In [14]:
lim_distinct_domains.count()

10000

Required to speed up analysis...

In [7]:
dmarc_mx_path = f"s3a://luvizottocesarg-tmp/oi-warehouse/testing/source=tranco/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}"

In [21]:
dmarc_w_mx_df.write.parquet(dmarc_mx_path)

In [10]:
dmarc_w_mx_df = spark.read.parquet(dmarc_mx_path)

In [11]:
# for limit to be deterministic, it's better to use sample(n, seed=1234).
dmarc_w_mx_df.count()
# limit + tranco of 2024,3,14:
# 43040
# limit + tranco of 2025,7,11:
# 62946

62946

In [38]:
dmarc_w_mx_df.groupBy("query_type").count().show()
# limit + tranco of 2024,3,14:
#+----------+-----+
#|query_type|count|
#+----------+-----+
#|        MX| 8910|
#|       TXT|19809|
#|         A|14321|
#+----------+-----+
# limit + tranco of 2025,7,11:
#+----------+-----+
#|query_type|count|
#+----------+-----+
#|        MX|11007|
#|       TXT|26477|
#|         A|19230|
#+----------+-----+

+----------+-----+
|query_type|count|
+----------+-----+
|        MX|11007|
|       TXT|26477|
|         A|19230|
+----------+-----+



#### How many of these are organizations, private users, and others?

In [41]:
#apx_ruas_rufs_df = dmarc_df
#apx_ruas_rufs_df = extract_rua_ruf_domains(dmarc_w_mx_df).groupBy("apex_domain"  # one domain may have multiple TXT records
#).agg(
#    psf.collect_set("rua_domains").alias("rua_domains_list"),   # list(list)
#    psf.collect_set("ruf_domains").alias("ruf_domains_list")  # list(list))
#)

In [11]:
rua_output = f"s3a://luvizottocesarg-tmp/oi-warehouse/aggregation/what=rua_addresses/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}"
ruf_output = f"s3a://luvizottocesarg-tmp/oi-warehouse/aggregation/what=ruf_addresses/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}"

In [87]:
# CAREFUL WHEN USING THE FULL DATASET. IT MAY NEED TO SAVE TO OBJSTORE AND THEN LOAD IT AGAIN...
aggr_rua_df = dmarc_df.select(psf.explode("rua_domains").alias("rua_domain")
).groupBy("rua_domain").count().withColumn(
    "rua_apex_domain", extract_apex_domain(psf.col("rua_domain"))
).groupBy("rua_apex_domain").agg(psf.sum(psf.col("count"))
).withColumn(
    "percent", psf.lit(100) * psf.col('sum(count)') / psf.sum('sum(count)').over(Window.partitionBy())
).sort("percent", ascending=False)
aggr_rua_df.write.parquet(rua_output)

aggr_ruf_df = dmarc_df.select(psf.explode("ruf_domains").alias("ruf_domain")
).groupBy("ruf_domain").count().withColumn(
    "ruf_apex_domain", extract_apex_domain(psf.col("ruf_domain"))
).groupBy("ruf_apex_domain").agg(psf.sum(psf.col("count"))
).withColumn(
    "percent", psf.lit(100) * psf.col('sum(count)') / psf.sum('sum(count)').over(Window.partitionBy())
).sort("percent", ascending=False)
aggr_ruf_df.write.parquet(ruf_output)

#output = f"s3a://luvizottocesarg-tmp/oi-warehouse/aggregation/what=rua_ruf_addresses/source={source}/year={start_date.year}/month={start_date.month:02d}/day={start_date.day:02d}"
#aggr_rua_ruf_df = aggr_rua_df.select(psf.col("rua_apex_domain").alias("rua_ruf_apex_domain"), psf.col("sum(count)")).union(aggr_ruf_df.select(psf.col("ruf_apex_domain").alias("rua_ruf_apex_domain"), psf.col("sum(count)"))
#).groupBy("rua_ruf_apex_domain").agg(psf.sum(psf.col("sum(count)"))
#).select("rua_ruf_apex_domain", psf.col("sum(sum(count))").alias("count")
#).withColumn(
#    "percent", psf.lit(100) * psf.col('count') / psf.sum('count').over(Window.partitionBy())
#).sort("percent", ascending=False)
#aggr_rua_ruf_df.write.parquet(output)

In [51]:
aggr_rua_pdf = spark.read.parquet(rua_output).toPandas()
aggr_ruf_pdf = spark.read.parquet(ruf_output).toPandas()

Py4JJavaError: An error occurred while calling o917.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 194.0 failed 4 times, most recent failure: Lost task 0.3 in stage 194.0 (TID 3162) (192.168.47.119 executor 5): TaskResultLost (result lost from block manager)
Driver stacktrace:
	at org.apache.spark.scheduler.DAGScheduler.failJobAndIndependentStages(DAGScheduler.scala:2856)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2(DAGScheduler.scala:2792)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2$adapted(DAGScheduler.scala:2791)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreach$(ResizableArray.scala:55)
	at scala.collection.mutable.ArrayBuffer.foreach(ArrayBuffer.scala:49)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:2791)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1

The final plot is in a local jupyter notebook.

How many times multiple mailto: is present in rua/ruf tags?

In [27]:
dmarc_df.withColumn(
    "nr_ruas", psf.size("rua_domains")
).groupBy("nr_ruas").count().show()

dmarc_df.withColumn(
    "nr_rufs", psf.size("ruf_domains")
).groupBy("nr_rufs").count().show()

+-------+------+
|nr_ruas| count|
+-------+------+
|      1|250111|
|      3|  2789|
|      4|   280|
|      2| 28889|
|      0|144379|
|      5|    26|
|      6|     7|
|      7|     2|
+-------+------+

+-------+------+
|nr_rufs| count|
+-------+------+
|      1|138607|
|      6|     1|
|      3|   558|
|      2|  9698|
|      0|277582|
|      4|    33|
|      5|     4|
+-------+------+



How many corner cases among all domains?

In [15]:
#bgazrt.hu BGAZRT.HU
#conferencenext.com mailinblue.com!10m
#crediautos.cl crediautos
#fruugo.com.tr
#mydwoje.pl dplads.com>
#pravslovo.ru pravslovo.ruadkim=r
#cambridgeone.org cambridge.o
#cfc.org.br tuxon.com.br._report._dmarc
#drogaraia.com.br DMARC.EVEREST.EMAIL
#gatefy.com mailto
#xiantao.gov.cn qiye.163.com.
#xtreme.bet xtreme.bet\
#anglesey.gov.uk ruf.aga
#astroved.com pct=100
case = 'agricolors.fr'

# THESE VARIABLES DOES NOT EXIST ANYMORE
dmarc_w_mx_df.filter(
    (psf.col("query_type") == "TXT")
    & (psf.col("query_name").startswith("_dmarc."))
    & (psf.col("query_name").contains(case))
).select("txt_text").show(truncate=False)

apx_ruas_rufs_df.filter(
    psf.col("apex_domain").contains(case)
).show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------+
|txt_text                                                                                                                         |
+---------------------------------------------------------------------------------------------------------------------------------+
|"v=DMARC1; p=none; sp=none; rua=mailto:dmarc@mailinblue.com!10m; ruf=mailto:dmarc@mailinblue.com!10m; rf=afrf; pct=100; ri=86400"|
|NULL                                                                                                                             |
+---------------------------------------------------------------------------------------------------------------------------------+

+--------------+----------------------+----------------------+
|apex_domain   |rua_addresses_list    |ruf_addresses_list    |
+--------------+----------------------+----------------------+
|agricolors.fr.|[[

In [13]:
# corner cases - with limit and tranco 2024:
# 1 - email domain has a typo
# a - missing TLD in one of the mailto
# apex_domain="crediautos.cl" -> "v=DMARC1; p=reject; fo=1; rua=mailto:dmarc_rua@emaildefense.proofpoint.com; ruf=mailto:dmarc_ruf@emaildefense.proofpoint.com;"
# b - characters (quotes) added to the rua/ruf domain name
# apex_domain="cambridgeone.org" -> "v=DMARC1; p=quarantine; rua=mailto:644b4767adf94d2a9f33fbe1b9072971@dmarc-reports.cloudflare.net,mailto:047acbdc29a7625@rep.dmarcanalyzer.com,mailto:dmarc-admin@cambridge.org; ruf=mailto:047acbdc29a7625@for.dmarcanalyzer.com,mailto:dmarc-admin@cambridge.o""rg; fo=1;"
# apex_domain="xtreme.bet" -> "\"v=DMARC1; p=reject; sp=reject; ruf=mailto:authfail@xtreme.bet; rua=mailto:aggrep@xtreme.bet\""
# apex_domain="anglesey.gov.uk" -> ruf=mailto:isle-of-anglesey-council@ruf.aga""ri-eu.com,mailto:dmarc-rua@dmarc.service.gov.uk
# c - misinterpretation of the standard
# apex_domain="cfc.org.br" -> "v=DMARC1;p=none;pct=100;rua=mailto:postmaster@mensageiro.tuxon.com.br,mailto:mensageiro@tuxon.com.br._report._dmarc"
# d - typo on mailto:
# apex_domain="gatefy.com" -> ruf=mailto:ed75d258-6313-11ea-bc55-0242ac130003@antifraud.gatefy.com,mailto:support@gatefy.com,mailto"":6a13379d44@ruf.easydmarc.com;"

# 2 - domain name is malformed according to RFC 5321 because of the syntax of domain name is not valid according to RFC 1035 sec. 2.3.1
# apex_domain="siter.io" -> ma""ilinblue.com!10m
# a - the quotation problem entail a different domain mailto:dmarc-ruf@pushwoosh.io,mailto:dmarc@pushwoosh.co  -- LIVE WITH IT!
# apex_domain="pushwoosh.com" -> "v=DMARC1; p=quarantine; sp=quarantine; pct=100; rua=mailto:c9312ab8@mxtoolbox.dmarc-report.com,mailto:dmarc-rua@pushwoosh.io,mailto:dmarc@pushwoosh.com; ruf=mailto:c9312ab8@forensics.dmarc-report.com,mailto:dmarc-ruf@pushwoosh.io,mailto:dmarc@pushwoosh.co""m; fo=1" 
# 3 - malformed txt record; additional quotes
# apex_domain="fruugo.com.tr" -> """v=DMARC1;""p=none;""sp=none;""pct=100;""ri=2419200;""rua=mailto:dmarc.reports@fruugo.com;""ruf=mailto:dmarc.reports@fruugo.com\"\""
# 4 - rua ending with > instead of only ;
# apex_domain="mydwoje.pl" -> "v=DMARC1; p=none; rua=mailto:dmarc@2zcfjaj2.uriports.com,mailto:dmarc@dplads.com>; ruf=mailto:dmarc@2zcfjaj2.uriports.com,dmarc@dplads.com>; fo=1:d:s"
# 5 - no ; between tags
# apex_domain="pravslovo.ru" -> "v=DMARC1; p=quarantine; rua=mailto:webmaster@pravslovo.ru; ruf=mailto:webmaster@pravslovo.ru adkim=r; aspf=s; pct=100; ri=86400; sp=none"
# apex_domain="astroved.com" -> "v=DMARC1;p=reject;rua=mailto:kameshvaran@astroved.com,pct=100;adkim=s;aspf=s"

queries = ["_dmarc.crediautos.cl.", 
           "_dmarc.cambridgeone.org.", 
           "_dmarc.xtreme.bet.", 
           "_dmarc.anglesey.gov.uk.", 
           "_dmarc.cfc.org.br.", 
           "_dmarc.gatefy.com.", 
           "_dmarc.siter.io.", 
           "_dmarc.pushwoosh.com.", 
           "_dmarc.fruugo.com.tr.", 
           "_dmarc.mydwoje.pl.", 
           "_dmarc.pravslovo.ru.", 
           "_dmarc.astroved.com."]

tranco_df.filter(
    (psf.col("query_type") == "TXT" )
    & (psf.col("query_name").isin(queries))
).select("query_name", "txt_text").show(truncate=False)

# ok and invalid labels coming from checkdmarc python library
domain_txt_dict = {
    'crediautos.cl': 'v=DMARC1; p=reject; fo=1; rua=mailto:dmarc_rua@emaildefense.proofpoint.com; ruf=mailto:dmarc_ruf@emaildefense.proofpoint.com;',  # ok
    'cambridgeone.org': 'v=DMARC1; p=quarantine; rua=mailto:644b4767adf94d2a9f33fbe1b9072971@dmarc-reports.cloudflare.net,mailto:047acbdc29a7625@rep.dmarcanalyzer.com,mailto:dmarc-admin@cambridge.org; ruf=mailto:047acbdc29a7625@for.dmarcanalyzer.com,mailto:dmarc-admin@cambridge.o""rg; fo=1;',  # invalid
    'xtreme.bet': '"v=DMARC1; p=reject; sp=reject; ruf=mailto:authfail@xtreme.bet; rua=mailto:aggrep@xtreme.bet"',  # ok
    'cfc.org.br': 'v=DMARC1; p=quarantine; fo=1; pct=5; rua=mailto:postmaster@cfc.org.br',  # pct value is less than 100. This leads to inconsistent and unpredictable policy enforcement
    'pushwoosh.com': 'v=DMARC1; p=quarantine; sp=quarantine; pct=100; rua=mailto:c9312ab8@mxtoolbox.dmarc-report.com,mailto:dmarc-rua@pushwoosh.io,mailto:dmarc@pushwoosh.com; ruf=mailto:c9312ab8@forensics.dmarc-report.com,mailto:dmarc-ruf@pushwoosh.io,mailto:dmarc@pushwoosh.co""m; fo=1',  # invalid
    'fruugo.com.tr': '""v=DMARC1;""p=none;""sp=none;""pct=100;""ri=2419200;""rua=mailto:dmarc.reports@fruugo.com;""ruf=mailto:dmarc.reports@fruugo.com""',  # invalid
    'siter.io': 'v=DMARC1; p=reject; p=quarantine; sp=none; rua=mailto:b6912d25d63842beb6825c439fe06625@dmarc-reports.cloudflare.net,mailto:ipm2sg0@ar.glockapps.com,mailto:re+af96ffc2ba6c@inbound.dmarcdigests.com,mailto:re+jv1mk3mupdi@dmarc.postmarkapp.com,mailto:dmarc@ma""ilinblue.com!10m; ruf=mailto:ipm2sg0@fr.glockapps.com,mailto:dmarc@mailinblue.com!10m; aspf=r; fo=1; rf=afrf; ri=86400;',  # invalid
    'gatefy.com': 'v=DMARC1; p=quarantine; pct=100; fo=1; ri=3600; rua=mailto:ed75d258-6313-11ea-bc55-0242ac130003@antifraud.gatefy.com,mailto:6a13379d44@rua.easydmarc.com; ruf=mailto:ed75d258-6313-11ea-bc55-0242ac130003@antifraud.gatefy.com,mailto:support@gatefy.com,mailto"":6a13379d44@ruf.easydmarc.com;',  # invalid
    'pravslovo.ru': 'v=DMARC1; p=quarantine; rua=mailto:webmaster@pravslovo.ru; ruf=mailto:webmaster@pravslovo.ru adkim=r; aspf=s; pct=100; ri=86400; sp=none',   # invalid - manually added
    'mydwoje.pl': 'v=DMARC1; p=none; rua=mailto:dmarc@2zcfjaj2.uriports.com,mailto:dmarc@dplads.com>; ruf=mailto:dmarc@2zcfjaj2.uriports.com,dmarc@dplads.com>; fo=1:d:s',  # invalid - manually added
}

+------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|query_name              |txt_text                                                                                                                                                                                                                                                                                                                                                                                  |
+------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [30]:
cases = ["genelec.ba", "somn.io", "jgean.com.br", "vesely-drak.cz", "rentrabb.it", "actens.io", "univsul.edu.iq", "mariowii.nl", "kamody.cz", 
         "payakumbuhkota.go.id", "ipboxcloud.com.br", "pinkfort.co", "green-edem.com.ua", "menfp.gouv.ht", "top4running.ro", "novissajoias.com.br", 
         "mosaico-cem.it", "le7sorelle.it", "vkino.com.ua", "netcon.com.br", "endlessmode.jp", "thechristmasshop.com.au", "hokutetsu.co.jp", "energieker.it", 
         "martens-tweewielers.nl", "augusta-staff.jp", "customs.go.jp", "empresaqui.com.br", "visor.ph", "fnspfdr.sk", "colosseumticket.cz", "k2a.ru", 
         "comfortflex.com.br", "svetbot.cz", "foxtrail.ch", "green-edem.com.ua", "roast.by", "internet.gr", "sellis-it.nl", "revi.io"]

extract_dmarc_tags(tranco_df).select("query_name", "rua_domains", "ruf_domains", "txt_text").withColumn(
    "apex_domain", extract_apex_domain(psf.col("query_name"))
).filter(
    psf.col("apex_domain").isin(cases)
).select("apex_domain", psf.explode_outer("rua_domains").alias("rua_domain"), psf.explode_outer("ruf_domains").alias("ruf_domain"), "txt_text"
).withColumn(
    "rua_apex_domain", extract_apex_domain(psf.col("rua_domain"))
).withColumn(
    "ruf_apex_domain", extract_apex_domain(psf.col("ruf_domain"))
).show(100, truncate=False)


+-----------------------+------------------------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+---------------+
|apex_domain            |rua_domain                                |ruf_domain                       |txt_text                                                                                                                                                                       |rua_apex_domain |ruf_apex_domain|
+-----------------------+------------------------------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+---------------+
|fnspfdr.sk             |hostcreators.sk hostcreators.sk        

### Graph analysis - general; attempt

#### How interconnected are third-party DMARC processors and domain owners?

In [89]:
print("\nTop 5 reporting destinations")
G.inDegrees.sort("inDegree", ascending=False).show(5, truncate=False)

self_loop_cnt = G.edges.filter(G.edges.src == G.edges.dst).count()
print(f"\nSelf-reporting domains - not external report: {self_loop_cnt}")

external_cnt = G.edges.filter(G.edges.src != G.edges.dst).count()
print(f"\nReports to an external domain: {external_cnt}")
#pagerank = G.pageRank(resetProbability=0.01, maxIter=20)
#pagerank.vertices.select("id", "pagerank").show(5, truncate=False)


Top 5 reporting destinations
+----------------------------+--------+
|id                          |inDegree|
+----------------------------+--------+
|proofpoint.com              |27464   |
|dmarc-reports.cloudflare.net|21680   |
|vali.email                  |15331   |
|dmarcanalyzer.com           |14837   |
|dmarcian.com                |14741   |
+----------------------------+--------+
only showing top 5 rows


Self-reporting domains - not external report: 187160

Reports to an external domain: 277354


In [15]:
# Search for pairs of vertices with edges in both directions between them.
motifs = G.find("(a)-[e]->(b); (b)-[e2]->(a)")
print(motifs.count())

302390


In [18]:
# Find chains of 4 vertices.
chain4 = G.find("(a)-[ab]->(b); (b)-[bc]->(c); (c)-[cd]->(d)")

# Query on sequence, with state (cnt)
#  (a) Define method for updating state given the next element of the motif.
def cum_edges(cnt, edge):
    edge_type = psf.col(edge)["edge_type"]
    return psf.when(edge_type == "rua", cnt + 1).otherwise(cnt)

#  (b) Use sequence operation to apply method to sequence of elements in motif.
#   In this case, the elements are the 3 edges.
edges = ["ab", "bc", "cd"]
num_edge_types = reduce(cum_edges, edges, psf.lit(0))
    
chainWith2Friends2 = chain4.withColumn("num_edges", num_edge_types).where(num_edge_types >= 2)
chainWith2Friends2.show()
print(chainWith2Friends2.count())

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------+
|                   a|                  ab|                   b|                  bc|                   c|                  cd|                   d|num_edges|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------+
|       {1-rk.com.ua}|{1-rk.com.ua, 1-r...|       {1-rk.com.ua}|{1-rk.com.ua, 1-r...|       {1-rk.com.ua}|{1-rk.com.ua, 1-r...|       {1-rk.com.ua}|        3|
|{1000grad-epaper.de}|{1000grad-epaper....|{1000grad-epaper.de}|{1000grad-epaper....|{1000grad-epaper.de}|{1000grad-epaper....|{1000grad-epaper.de}|        3|
| {100directions.com}|{100directions.co...| {100directions.com}|{100directions.co...| {100directions.com}|{100directions.co...| {100directions.com}|        3|
|          {100sp.ru}|{100sp.ru, 100sp....|   

In [19]:
# "detecting communities in networks.
# Each node in the network is initially assigned to its own community.
# At every superstep, nodes send their community affiliation to all neighbors and update their state to the most frequent community affiliation of incoming messages."
# https://docs.databricks.com/aws/en/notebooks/source/graphframes-user-guide-py.html
G.labelPropagation(maxIter=5).show()

+--------------------+-------------+
|                  id|        label|
+--------------------+-------------+
|  1000grad-epaper.de|            0|
|      SiouxChief.com| 231928235824|
|        abstract.com|           26|
| ac-aix-marseille.fr|           29|
|    adventsource.org|           54|
|            agent.ru|           65|
|          ampfit.com|          112|
|           amsnet.pl|          113|
|     artofthepot.com|          155|
|     astrocenter.com|          167|
|     azamgarh.nic.in|          191|
|     basicincome.org|          222|
|        betterce.com| 841813592038|
|  blogdomagno.com.br|          270|
|         blynk.cloud|  51539607825|
|         bnbcash.app|          278|
|     bookeasy.com.au| 678604833317|
|          bosfera.ru|          293|
|boutiqueauxmainsd...|   8589936674|
|      brainmarket.cz|1382979469562|
+--------------------+-------------+
only showing top 20 rows



In [11]:
G.triangleCount().sort("count", ascending=False).show(truncate=False)

+-----+----------------------------+
|count|id                          |
+-----+----------------------------+
|1056 |vali.email                  |
|706  |proofpoint.com              |
|262  |playtika.com                |
|238  |dmarcian.com                |
|227  |dhs.gov                     |
|201  |dmarc-reports.cloudflare.net|
|177  |postmarkapp.com             |
|135  |dmarcadvisor.com            |
|126  |dmarcdigests.com            |
|120  |agari.com                   |
|118  |dmarcanalyzer.com           |
|116  |powerdmarc.com              |
|99   |ondmarc.com                 |
|91   |dmarc-report.com            |
|69   |easydmarc.us                |
|67   |gm.com                      |
|65   |dmarc.gov.sa                |
|65   |wolterskluwer.com           |
|60   |dyson.com                   |
|60   |redsift.cloud               |
+-----+----------------------------+
only showing top 20 rows



## RQ2 - What is the geographical distribution of DMARC report receivers, domain owners and their email servers, and what are the potential implications of cross jurisdictional data transit?

#### Are there differences on how types of domains use rua/ruf tags? Is .gov different than .org, .net or .com?

#### What about the differences on ccTLDs?

#### What is the proportion of domains that outsource DMARC processing to another country?

#### To what extent the top DMARC processing organizations adhere to relevant data protection and privacy regulations when handling report data from domain owners of different jurisdictions?

## RQ3 - How DMARC report receivers evolved over the course of 1 year?

#### Has usage of ruf tags changed over time?

#### Has dependency on third-party processors increased or decreased over time?

#### How frequent these dependencies change over time?

## Epilogue

In [ ]:
clean_spark()

## Old/unused code

##### Using NetworkX -- deprecated

In [18]:
# using NetworkX
def create_graph_from_pyspark_df(df, good_error_domains):
    """
    Convert PySpark DataFrame to NetworkX graph
    """
    # Convert to Pandas for easier graph creation
    pdf = df.toPandas()

    # Create directed graph
    domain_graph = nx.DiGraph()

    # Add nodes (all unique domains)
    all_domains = set()
    for _, row in pdf.iterrows():
        ruas = row['rua_domains_list'] if row['rua_domains_list'] else [[]]
        rufs = row['ruf_domains_list'] if row['ruf_domains_list'] else [[]]
        flat_ruas_rufs = set(list(itertools.chain(*(ruas+rufs))))

        apex = row['apex_domain'].rstrip('.')
        all_domains.add(apex)

        # below is for statistics
        good_rua_ruf_tags_cnt = 0
        bad_rua_ruf_tags_cnt = 0
        for domain in flat_ruas_rufs:
            if not domain or domain == "":
                continue
            def match_domain(pattern, domain):
                m = re.search(pattern, domain, re.IGNORECASE)
                if m:
                    apex_domain = f"{m.group(1)}.{m.group(2)}"
                    all_domains.add(apex_domain)
                return m

            if match_domain(domain_pattern, domain):
                good_rua_ruf_tags_cnt += 1
            else:
                bad_rua_ruf_tags_cnt += 1

        # statistics
        good_error_domains[apex] = [good_rua_ruf_tags_cnt, bad_rua_ruf_tags_cnt]

    domain_graph.add_nodes_from(all_domains)

    # Add edges for RUA/RUF relationships
    def add_edges(domain_graph, pdf, rua_ruf_col, edge_type):
        for _, row in pdf.iterrows():
            apex = row['apex_domain'].rstrip('.')
            rua_ruf_list = row[rua_ruf_col] if row[rua_ruf_col] else [[]]
            rua_ruf = set(list(itertools.chain(*rua_ruf_list)))

            for domain in rua_ruf:
                if not domain or domain == "":
                    continue
                m = re.search(domain_pattern, domain, re.IGNORECASE)
                if m:
                    rua_ruf_apex = f"{m.group(1)}.{m.group(2)}"
                    domain_graph.add_edge(apex, rua_ruf_apex, edge_type=edge_type, label=edge_type.upper())


    add_edges(domain_graph, pdf, "rua_domains_list", "rua")

    add_edges(domain_graph, pdf, "ruf_domains_list", "ruf")

    return domain_graph

# Create the graph
good_error_domains = defaultdict(list)
G = create_graph_from_pyspark_df(apx_ruas_rufs_df, good_error_domains)

In [ ]:
# for NetworkX:
if False:
    print(f"Number of nodes: {G.number_of_nodes()}")
    print(f"Number of edges: {G.number_of_edges()}")
    print(f"Is directed: {G.is_directed()}")
    
    print("nr of apex domains in the graph:")
    print(len(good_error_domains.keys()))
    
    print("nr of domains with one or more errors:")
    one_or_more_tags_wrong_cnt = 0
    for domain, cnt_list in good_error_domains.items():
        good_cnt = cnt_list[0]
        bad_cnt = cnt_list[1]
        if bad_cnt > 0:
            one_or_more_tags_wrong_cnt += 1
    print(one_or_more_tags_wrong_cnt)
    
    # how many have all rua/ruf with errors
    print("how many apex domains have all tags wrong (ie won't be sending reports anyway)?")
    all_tags_wrong_cnt = 0
    for domain, cnt_list in good_error_domains.items():
        good_cnt = cnt_list[0]
        bad_cnt = cnt_list[1]
        if good_cnt == 0 and bad_cnt > 0:
            all_tags_wrong_cnt += 1
    print(all_tags_wrong_cnt)
    # tranco 2024,3,14 with limit=10000
    #Number of nodes: 4042
    #Number of edges: 2438
    #Is directed: True
    #nr of apex domains in the graph:
    #3597
    #nr of domains with one or more errors:
    #67
    #how many apex domains have all tags wrong (ie won't be sending reports anyway)?
    #50
    #

# NetworkX
if False:
    # Find nodes with highest in-degree (most popular reporting destinations)
    in_degrees = dict(G.in_degree())
    top_reporting_destinations = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:5]
    print("\nTop 5 reporting destinations - nr of domains resporting to these:")
    for domain, count in top_reporting_destinations:
        print(f"  {domain}: {count}")
    
    # Find self-reporting domains (domains that report to themselves)
    self_reporting = [node for node in G.nodes() if G.has_edge(node, node)]
    print(f"\nSelf-reporting domains - not external report: {len(self_reporting)}")
    
    external_reporting = [node for node in G.nodes() if not G.has_edge(node, node)]
    print(f"\nReports to an external domain: {len(external_reporting)}")
    # tranco 2024,3,14; limit=10000
    #Top 5 reporting destinations - nr of domains resporting to these:
    #  dmarc-reports.cloudflare.net: 135
    #  proofpoint.com: 101
    #  vali.email: 100
    #  dmarcian.com: 76
    #  dmarcanalyzer.com: 68

    #Self-reporting domains - not external report: 1099

    #Reports to an external domain: 2943
    #

In [55]:
# Visualization function
def visualize_graph(G, figsize=(15, 10)):
    """
    Visualize the graph with different colors for different edge types
    """
    plt.figure(figsize=figsize)
    
    # Create layout
    pos = nx.spring_layout(G, k=3, iterations=50)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', 
                          node_size=300, alpha=0.7)
    
    # Draw RUA edges in blue
    rua_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'rua']
    nx.draw_networkx_edges(G, pos, edgelist=rua_edges, 
                          edge_color='blue', alpha=0.6, width=1.5, 
                          arrowsize=20, label='RUA (Aggregate)')
    
    # Draw RUF edges in red
    ruf_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'ruf']
    nx.draw_networkx_edges(G, pos, edgelist=ruf_edges, 
                          edge_color='red', alpha=0.6, width=1.5, 
                          arrowsize=20, label='RUF (Failure)')
    
    # Draw labels for important nodes only (to avoid clutter)
    important_nodes = {node: node for node, degree in in_degrees.items() if degree > 1}
    nx.draw_networkx_labels(G, pos, important_nodes, font_size=8)
    
    plt.title("DMARC Reporting Relationships Graph")
    plt.legend()
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Create and show the visualization
visualize_graph(G)

In [109]:
# Export NetworkX graph data for external tools
def export_graph_data(G, filename_prefix="dmarc_graph"):
    """
    Export graph data in various formats
    """
    # Export as GraphML (can be opened in Gephi, Cytoscape, etc.)
    nx.write_graphml(G, f"{filename_prefix}.graphml")
    
    # Export edge list as CSV
    edges_df = pd.DataFrame(G.edges(data=True))
    edges_df.to_csv(f"{filename_prefix}_edges.csv", index=False)
    
    # Export node list as CSV
    nodes_df = pd.DataFrame({"node": list(G.nodes())})
    nodes_df.to_csv(f"{filename_prefix}_nodes.csv", index=False)
    
    print(f"Graph exported as {filename_prefix}.graphml")
    print(f"Edges exported as {filename_prefix}_edges.csv")
    print(f"Nodes exported as {filename_prefix}_nodes.csv")

export_graph_data(G)

Graph exported as dmarc_graph.graphml
Edges exported as dmarc_graph_edges.csv
Nodes exported as dmarc_graph_nodes.csv
